# Train Sentiment Overall Model bằng PhoBERT

Notebook này dùng để train model đánh giá **cảm xúc tổng thể của cuộc hội thoại** giữa:

- CSKH và khách hàng
- Chatbot và khách hàng

Pipeline gồm:

1. Load dữ liệu sentiment từ `data/processed/positivity`
2. Tiền xử lý dữ liệu
3. Chuẩn hóa nhãn sentiment
4. Tokenize bằng PhoBERT tokenizer
5. Build model PhoBERT classification
6. Train model
7. Validation/Test
8. Nhập thử hội thoại và dự đoán cảm xúc

Output model sẽ được lưu vào:

```text
models/sentiment_phobert/final_model/
```

## 1. Cài thư viện

Chạy cell này trước.

Nếu bạn chạy local và đã cài thư viện rồi thì có thể bỏ qua.

In [1]:
# Chạy 1 lần nếu thiếu thư viện hoặc lỗi "Numpy is not available":
# !pip install -r requirements.txt -q

## 2. Import thư viện và cấu hình

Bạn có thể chỉnh các biến như `DATA_DIR`, `MODEL_NAME`, `NUM_EPOCHS`, `BATCH_SIZE`.

In [2]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"  # tránh lỗi tải model qua XET storage
import re
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)


def check_runtime():
    """torch 2.2 + numpy 2.x sẽ gây lỗi 'Numpy is not available' khi evaluate."""
    np_major = int(np.__version__.split(".")[0])
    torch_parts = [int(x) for x in torch.__version__.split("+")[0].split(".")[:2]]
    torch_major, torch_minor = torch_parts[0], torch_parts[1]

    if np_major >= 2 and torch_major == 2 and torch_minor < 6:
        raise RuntimeError(
            f"numpy {np.__version__} + torch {torch.__version__} không tương thích.\n"
            "Chạy trong terminal: pip install -r requirements.txt\n"
            "Sau đó restart kernel notebook."
        )

    # Kiểm tra bridge torch <-> numpy (lỗi hay gặp khi evaluate)
    _ = torch.tensor([1.0]).numpy()


check_runtime()
print("NumPy:", np.__version__)
print("Torch version:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
print("MPS available:", torch.backends.mps.is_available())

/Users/thdziii/Documents/Github/CareScore-AI/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NumPy: 1.26.4
Torch version: 2.2.2
Transformers: 4.57.6
CUDA available: False
MPS available: True


In [3]:
# ============================================================
# CONFIG
# ============================================================

DATA_DIR = Path("data/processed/positivity")
OUTPUT_DIR = Path("models/sentiment_phobert")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# PhoBERT phù hợp cho tiếng Việt.
# Nếu bị lỗi tải model, chạy: python download_model.py
HF_MODEL_ID = "vinai/phobert-base"
LOCAL_MODEL_DIR = Path("models/phobert-base")
_has_safetensors = (LOCAL_MODEL_DIR / "model.safetensors").exists()
_has_pytorch_bin = (LOCAL_MODEL_DIR / "pytorch_model.bin").exists()
MODEL_NAME = str(LOCAL_MODEL_DIR) if (_has_safetensors or _has_pytorch_bin) else HF_MODEL_ID
USE_SAFETENSORS = _has_safetensors  # torch<2.6 bắt buộc dùng safetensors thay vì .bin

MAX_LENGTH = 256
TEST_SIZE = 0.15
VALID_SIZE = 0.15
RANDOM_SEED = 42

NUM_EPOCHS = 3
BATCH_SIZE = 2  # giảm để tránh OOM trên Mac MPS
GRADIENT_ACCUMULATION_STEPS = 4  # effective batch = 2 x 4 = 8
USE_MPS_DEVICE = False  # Mac MPS dễ hết RAM khi fine-tune PhoBERT
RESUME_TRAINING = True  # tiếp tục từ checkpoint nếu có
LEARNING_RATE = 2e-5

id2label = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2,
}

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MODEL_NAME:", MODEL_NAME)
if MODEL_NAME == HF_MODEL_ID:
    print("⚠️  Chưa có models/phobert-base/pytorch_model.bin — sẽ tải từ Hugging Face.")
    print("   Nếu lỗi mạng, chạy: python download_model.py")
else:
    print("✓ Dùng model local:", LOCAL_MODEL_DIR)
    if _has_safetensors:
        print("✓ Weights: model.safetensors")
    elif _has_pytorch_bin:
        print("⚠️  Chỉ có pytorch_model.bin — nên chạy python download_model.py để tạo safetensors")

DATA_DIR: data/processed/positivity
OUTPUT_DIR: models/sentiment_phobert
MODEL_NAME: models/phobert-base
✓ Dùng model local: models/phobert-base
✓ Weights: model.safetensors


## 3. Thiết lập seed

Giúp kết quả train ổn định hơn giữa các lần chạy.

In [4]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)

## 4. Tiền xử lý text

Với BERT/PhoBERT, không nên xóa quá nhiều thông tin.  
Ta chỉ chuẩn hóa nhẹ:

- Chuyển chữ thường
- Thay URL bằng `<url>`
- Thay email bằng `<email>`
- Thay số điện thoại bằng `<phone>`
- Xóa khoảng trắng thừa

Với hội thoại, có thể giữ format như:

```text
Khách hàng: ...
Nhân viên: ...
```

In [5]:
def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = text.lower()

    # Thay URL, email, số điện thoại bằng token chung
    text = re.sub(r"http\S+|www\S+", " <url> ", text)
    text = re.sub(r"\S+@\S+", " <email> ", text)
    text = re.sub(r"\b\d{9,11}\b", " <phone> ", text)

    # Chuẩn hóa khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()

    return text


def build_conversation_text(row):
    '''
    Nếu sau này dữ liệu có các cột riêng cho khách hàng/nhân viên,
    hàm này sẽ ghép lại thành một đoạn hội thoại.
    '''
    possible_agent_cols = [
        "agent_text", "staff_text", "employee_text", 
        "bot_text", "assistant_text", "nhan_vien"
    ]
    possible_customer_cols = [
        "customer_text", "user_text", "client_text", 
        "khach_hang"
    ]

    agent_parts = []
    customer_parts = []

    for col in possible_agent_cols:
        if col in row and pd.notna(row[col]):
            agent_parts.append(str(row[col]))

    for col in possible_customer_cols:
        if col in row and pd.notna(row[col]):
            customer_parts.append(str(row[col]))

    if agent_parts or customer_parts:
        conversation = ""
        if customer_parts:
            conversation += "Khách hàng: " + " ".join(customer_parts) + " "
        if agent_parts:
            conversation += "Nhân viên: " + " ".join(agent_parts)
        return conversation.strip()

    return ""

## 5. Hàm tìm cột text và label

Mỗi dataset có thể đặt tên cột khác nhau, ví dụ:

- `text`
- `comment`
- `sentence`
- `review`
- `content`

Code sẽ tự tìm cột phù hợp.

In [6]:
TEXT_COL_CANDIDATES = [
    "text",
    "comment",
    "sentence",
    "review",
    "content",
    "utterance",
    "conversation",
    "conversation_text",
    "question",
    "answer",
]

LABEL_COL_CANDIDATES = [
    "label",
    "sentiment",
    "sentiment_label",
    "class",
    "target",
]


def find_column(columns, candidates):
    columns_lower = {c.lower(): c for c in columns}

    for cand in candidates:
        if cand.lower() in columns_lower:
            return columns_lower[cand.lower()]

    return None

## 6. Chuẩn hóa nhãn sentiment

Model sẽ học 3 nhãn:

```text
0 = negative
1 = neutral
2 = positive
```

Nếu dataset có nhãn dạng 1-5, notebook sẽ map như sau:

```text
1, 2 → negative
3    → neutral
4, 5 → positive
```

In [7]:
def map_label_to_sentiment(label):
    '''
    Chuẩn hóa nhãn về 3 lớp:
    0 = negative
    1 = neutral
    2 = positive
    '''
    if pd.isna(label):
        return None

    raw = str(label).strip().lower()

    # Label dạng text
    if raw in ["negative", "neg", "negative sentiment", "tiêu cực", "tieu cuc"]:
        return 0
    if raw in ["neutral", "neu", "trung lập", "trung lap"]:
        return 1
    if raw in ["positive", "pos", "positive sentiment", "tích cực", "tich cuc"]:
        return 2

    # Label dạng số
    try:
        value = int(float(raw))

        # Trường hợp nhãn 0,1,2
        if value in [0, 1, 2]:
            return value

        # Trường hợp nhãn 1-5
        if value in [1, 2]:
            return 0
        if value == 3:
            return 1
        if value in [4, 5]:
            return 2

    except Exception:
        return None

    return None

## 7. Load toàn bộ dữ liệu sentiment

Notebook sẽ tự đọc tất cả file `.csv` trong:

```text
data/processed/positivity
```

Nếu bạn chạy Colab, hãy upload folder `data/` hoặc mount Google Drive trước.

In [8]:
def read_csv_robust(csv_path):
    try:
        return pd.read_csv(csv_path)
    except Exception:
        return pd.read_csv(csv_path, engine="python", on_bad_lines="skip")


def load_all_sentiment_data(data_dir: Path):
    csv_files = list(data_dir.rglob("*.csv"))

    rows = []

    print(f"Tìm thấy {len(csv_files)} file CSV trong {data_dir}")

    for csv_path in csv_files:
        # Bỏ qua file không phải dữ liệu train
        name_lower = csv_path.name.lower()
        if "dataset_info" in name_lower:
            continue
        if "annotation_template" in name_lower:
            continue
        if "summary" in name_lower:
            continue

        try:
            df = read_csv_robust(csv_path)
        except Exception as e:
            print(f"Không đọc được file: {csv_path} | lỗi: {e}")
            continue

        text_col = find_column(df.columns, TEXT_COL_CANDIDATES)
        label_col = find_column(df.columns, LABEL_COL_CANDIDATES)

        if text_col is None:
            print(f"Bỏ qua file vì không tìm thấy cột text: {csv_path}")
            print(f"Các cột hiện có: {list(df.columns)}")
            continue

        if label_col is None:
            print(f"Bỏ qua file vì không tìm thấy cột label: {csv_path}")
            print(f"Các cột hiện có: {list(df.columns)}")
            continue

        print(f"Đọc file: {csv_path}")
        print(f"  Text column : {text_col}")
        print(f"  Label column: {label_col}")
        print(f"  Rows        : {len(df)}")

        for _, row in df.iterrows():
            text = str(row[text_col])

            # Nếu có cột hội thoại riêng thì ghép lại
            conversation_text = build_conversation_text(row)
            if conversation_text:
                text = conversation_text

            label = map_label_to_sentiment(row[label_col])
            if label is None:
                continue

            text = normalize_text(text)

            if len(text) < 2:
                continue

            rows.append(
                {
                    "text": text,
                    "label": label,
                    "source_file": str(csv_path),
                }
            )

    data = pd.DataFrame(rows)

    if data.empty:
        raise ValueError(
            "Không load được dữ liệu sentiment nào. "
            "Hãy kiểm tra lại DATA_DIR và tên cột trong file CSV."
        )

    # Xóa duplicate để tránh train/test bị trùng
    data = data.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)

    print("\nTổng số dòng sau khi load:", len(data))
    print("\nPhân bố nhãn:")
    print(data["label"].value_counts().sort_index())
    print("\nMapping:")
    print(id2label)

    return data


df = load_all_sentiment_data(DATA_DIR)
df.head()

Tìm thấy 2 file CSV trong data/processed/positivity
Đọc file: data/processed/positivity/test/test.csv
  Text column : text
  Label column: sentiment
  Rows        : 2224
Đọc file: data/processed/positivity/train/train.csv
  Text column : text
  Label column: sentiment
  Rows        : 7786

Tổng số dòng sau khi load: 10003

Phân bố nhãn:
label
0    2405
1    1270
2    6328
Name: count, dtype: int64

Mapping:
{0: 'negative', 1: 'neutral', 2: 'positive'}


,text,label,source_file
0,"điện thoải ổn. facelock cực nhanh, vân tay ôk ...",2,data/processed/positivity/test/test.csv
1,"mình mới mua vivo91c. tải ứng dụng ,games nhan...",2,data/processed/positivity/test/test.csv
2,xấu đẹp gì ko biết nhưng rất ưng tgdđ phục vụ ...,2,data/processed/positivity/test/test.csv
3,màn hình hơi lác khi chơi game. game nặng thì ...,2,data/processed/positivity/test/test.csv
4,"nói chung máy đẹp với màn amoled, ổn trong tầm...",2,data/processed/positivity/test/test.csv


## 8. Kiểm tra nhanh dữ liệu

Cell này giúp bạn xem dữ liệu đã chuẩn hóa về dạng `text, label` chưa.

In [9]:
print("Kích thước dữ liệu:", df.shape)
display(df.head(10))

print("\nSố lượng theo nhãn:")
display(df["label"].map(id2label).value_counts())

Kích thước dữ liệu: (10003, 3)


,text,label,source_file
0,"điện thoải ổn. facelock cực nhanh, vân tay ôk ...",2,data/processed/positivity/test/test.csv
1,"mình mới mua vivo91c. tải ứng dụng ,games nhan...",2,data/processed/positivity/test/test.csv
2,xấu đẹp gì ko biết nhưng rất ưng tgdđ phục vụ ...,2,data/processed/positivity/test/test.csv
3,màn hình hơi lác khi chơi game. game nặng thì ...,2,data/processed/positivity/test/test.csv
4,"nói chung máy đẹp với màn amoled, ổn trong tầm...",2,data/processed/positivity/test/test.csv
5,"đt có 3 lỗi: 1 là chế dô bỏ túi, 2 là ánh sáng...",2,data/processed/positivity/test/test.csv
6,thật tuyệt máy qua mượt túi thích như thế giới...,2,data/processed/positivity/test/test.csv
7,"sản phẩm tốt, pin sài cả ngày chưa hết, chơi g...",2,data/processed/positivity/test/test.csv
8,mua đc 1 tuần và đt sài khá tốt.. giá tầm trun...,2,data/processed/positivity/test/test.csv
9,pin nhanh hết vân tay cũng chưa đk nhạy nhân v...,0,data/processed/positivity/test/test.csv



Số lượng theo nhãn:


label
positive    6328
negative    2405
neutral     1270
Name: count, dtype: int64

## 9. Chia train / validation / test

Dữ liệu được chia thành:

- Train: dùng để model học
- Validation: dùng để theo dõi trong lúc train
- Test: dùng để đánh giá cuối cùng

Notebook dùng `stratify` để giữ tỷ lệ nhãn tương đối đều.

In [10]:
def split_data(df):
    train_df, temp_df = train_test_split(
        df,
        test_size=TEST_SIZE + VALID_SIZE,
        random_state=RANDOM_SEED,
        stratify=df["label"],
    )

    relative_test_size = TEST_SIZE / (TEST_SIZE + VALID_SIZE)

    valid_df, test_df = train_test_split(
        temp_df,
        test_size=relative_test_size,
        random_state=RANDOM_SEED,
        stratify=temp_df["label"],
    )

    print("Kích thước dữ liệu:")
    print("Train:", len(train_df))
    print("Valid:", len(valid_df))
    print("Test :", len(test_df))

    return train_df.reset_index(drop=True), valid_df.reset_index(drop=True), test_df.reset_index(drop=True)


train_df, valid_df, test_df = split_data(df)

display(train_df.head())

Kích thước dữ liệu:
Train: 7002
Valid: 1500
Test : 1501


,text,label,source_file
0,mới mua đc 1 ngày sản phẩm ok trong tầm giá nh...,2,data/processed/positivity/test/test.csv
1,moi thứ đều rất ok. nói chung là mới xài đc 3 ...,2,data/processed/positivity/test/test.csv
2,máy sài ngon ko giống như mấy bạn đánh giá. ch...,2,data/processed/positivity/test/test.csv
3,máy em chơi game rất hay bị khựng cho em hỏi l...,2,data/processed/positivity/train/train.csv
4,iphone 7plus không thể phát nhạc cho hai thiết...,1,data/processed/positivity/train/train.csv


## 10. Tokenize dữ liệu bằng PhoBERT tokenizer

Tokenize là bước biến text thành số để model có thể hiểu.

In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

train_dataset = Dataset.from_pandas(train_df[["text", "label"]])
valid_dataset = Dataset.from_pandas(valid_df[["text", "label"]])
test_dataset = Dataset.from_pandas(test_df[["text", "label"]])

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
valid_dataset = valid_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
valid_dataset = valid_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

train_dataset.set_format("torch")
valid_dataset.set_format("torch")
test_dataset.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_dataset)
print(valid_dataset)
print(test_dataset)

Map: 100%|██████████| 1501/1501 [00:00<00:00, 2170.11 examples/s]

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 7002
})
Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1500
})
Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1501
})


In [12]:
import sys
import torch
import transformers

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

Python executable: /Users/thdziii/Documents/Github/CareScore-AI/.venv/bin/python
Python version: 3.12.5 (v3.12.5:ff3bc82f7c9, Aug  7 2024, 05:32:06) [Clang 13.0.0 (clang-1300.0.29.30)]
Torch: 2.2.2
Transformers: 4.57.6
CUDA: False
CUDA version: None


## 11. Build model PhoBERT

Ta dùng `AutoModelForSequenceClassification`, tức là PhoBERT + một classification head.

In [13]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
    use_safetensors=USE_SAFETENSORS,
)

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()  # tiết kiệm RAM khi train
    model.config.use_cache = False

print(model.config)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at models/phobert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaConfig {
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "negative",
    "1": "neutral",
    "2": "positive"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "negative": 0,
    "neutral": 1,
    "positive": 2
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 258,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "tokenizer_class": "PhobertTokenizer",
  "transformers_version": "4.57.6",
  "type_vocab_size": 1,
  "use_cache": false,
  "vocab_size": 64001
}



## 12. Metrics đánh giá

Sử dụng:

- Accuracy
- Precision
- Recall
- F1-score

Với dữ liệu lệch nhãn, F1-score thường quan trọng hơn accuracy.

In [14]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if hasattr(logits, "detach"):
        logits = logits.detach().cpu().float().numpy()
    if hasattr(labels, "detach"):
        labels = labels.detach().cpu().numpy()
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted",
        zero_division=0,
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision_weighted": precision,
        "recall_weighted": recall,
        "f1_weighted": f1,
    }

## 13. Train model

Nếu bạn dùng CPU, bước này sẽ khá lâu.  
Nếu có GPU, hãy bật GPU trong Colab:

```text
Runtime → Change runtime type → GPU
```

In [15]:
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    logging_dir=str(OUTPUT_DIR / "logs"),
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    use_mps_device=USE_MPS_DEVICE,
    use_cpu=not torch.cuda.is_available() and not USE_MPS_DEVICE,
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

_checkpoints = sorted(OUTPUT_DIR.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
_resume = str(_checkpoints[-1]) if (RESUME_TRAINING and _checkpoints) else None
if _resume:
    print("Resume từ:", _resume)

trainer.train(resume_from_checkpoint=_resume)


Epoch,Training Loss,Validation Loss,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted
1,0.541300,0.521190,0.787333,0.787925,0.787333,0.783750
2,0.475100,0.545175,0.796667,0.777752,0.796667,0.776660
3,0.397500,0.590072,0.804000,0.799988,0.804000,0.801586


TrainOutput(global_step=2628, training_loss=0.4906050919397781, metrics={'train_runtime': 23312.584, 'train_samples_per_second': 0.901, 'train_steps_per_second': 0.113, 'total_flos': 642631911491304.0, 'train_loss': 0.4906050919397781, 'epoch': 3.0})

## 14. Validation/Test

Đánh giá model trên tập test chưa được dùng trong lúc train.

In [16]:
print("Đánh giá trên tập test:")
test_result = trainer.evaluate(test_dataset)
print(test_result)

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print("\nClassification report:")
print(
    classification_report(
        labels,
        preds,
        target_names=[id2label[i] for i in range(3)],
        zero_division=0,
    )
)

output_test_path = OUTPUT_DIR / "test_predictions.csv"

result_df = test_df.copy()
result_df["pred_label_id"] = preds
result_df["pred_label"] = [id2label[int(p)] for p in preds]
result_df["true_label"] = [id2label[int(y)] for y in labels]

result_df.to_csv(output_test_path, index=False, encoding="utf-8-sig")

print(f"Đã lưu dự đoán test tại: {output_test_path}")
display(result_df.head(20))

Đánh giá trên tập test:


{'eval_loss': 0.6045312881469727, 'eval_accuracy': 0.7854763491005996, 'eval_precision_weighted': 0.785333117917907, 'eval_recall_weighted': 0.7854763491005996, 'eval_f1_weighted': 0.7848404403087006, 'eval_runtime': 362.1147, 'eval_samples_per_second': 4.145, 'eval_steps_per_second': 2.074, 'epoch': 3.0}

Classification report:
              precision    recall  f1-score   support

    negative       0.78      0.70      0.74       361
     neutral       0.32      0.34      0.33       191
    positive       0.88      0.91      0.89       949

    accuracy                           0.79      1501
   macro avg       0.66      0.65      0.65      1501
weighted avg       0.79      0.79      0.78      1501

Đã lưu dự đoán test tại: models/sentiment_phobert/test_predictions.csv


,text,label,source_file,pred_label_id,pred_label,true_label
0,"máy ổn , nhưng wf máy mình rất tệ đang chơi ga...",1,data/processed/positivity/test/test.csv,1,neutral,neutral
1,mọi thứ đều ổn ngoại trừ pin.em mới mua về ngà...,2,data/processed/positivity/test/test.csv,2,positive,positive
2,"mua hơn 6 tháng. máy pin trâu, xài ok, mình bá...",2,data/processed/positivity/train/train.csv,2,positive,positive
3,mới lấy hôm 28/7 theo mình máy cầm chắc tay. p...,2,data/processed/positivity/test/test.csv,2,positive,positive
4,"vừa mua lúc chiều, máy mượt, vân tay nhạy,pin ...",2,data/processed/positivity/train/train.csv,2,positive,positive
5,sản phẩm rất ngon trong tầm giá 7-9tr. đã mua ...,2,data/processed/positivity/train/train.csv,2,positive,positive
6,"mình mới mua ngày 04/8 pin sài khá lâu, chơi g...",2,data/processed/positivity/train/train.csv,2,positive,positive
7,máy hay nóng tự đóng ứng dụng pin sạc nhanh ko...,0,data/processed/positivity/train/train.csv,0,negative,negative
8,mình chỉ mới mua về chưa được một tháng mà bị ...,0,data/processed/positivity/test/test.csv,0,negative,negative
9,"quá ngon,pin trâu,sạc 2h đầy rồi,liên quân bật...",2,data/processed/positivity/test/test.csv,2,positive,positive


## 15. Lưu model

Sau khi train xong, model sẽ được lưu vào:

```text
models/sentiment_phobert/final_model/
```

In [17]:
final_dir = OUTPUT_DIR / "final_model"
final_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))

with open(final_dir / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "id2label": id2label,
            "label2id": label2id,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print(f"Đã lưu model tại: {final_dir}")

Đã lưu model tại: models/sentiment_phobert/final_model


## 16. Hàm dự đoán cảm xúc cho hội thoại mới

Bạn có thể nhập một đoạn hội thoại CSKH - khách hàng hoặc chatbot - khách hàng.

In [18]:
def predict_sentiment(text, model, tokenizer, id2label):
    model.eval()

    normalized = normalize_text(text)

    inputs = tokenizer(
        normalized,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]
        pred_id = int(torch.argmax(probs).item())

    return {
        "input": text,
        "normalized_input": normalized,
        "label": id2label[pred_id],
        "label_id": pred_id,
        "confidence": float(probs[pred_id].item()),
        "probabilities": {
            id2label[i]: float(probs[i].item())
            for i in range(len(id2label))
        },
    }

## 17. Test thử bằng hội thoại mẫu

In [19]:
examples = [
    '''
    Khách hàng: Tôi đặt hàng 5 ngày rồi vẫn chưa nhận được, rất bực mình.
    Nhân viên: Dạ em xin lỗi anh/chị vì sự chậm trễ này. Em sẽ kiểm tra đơn hàng và phản hồi lại ngay ạ.
    ''',
    '''
    Khách hàng: Cảm ơn shop, nhân viên hỗ trợ rất nhanh và nhiệt tình.
    Nhân viên: Dạ em cảm ơn anh/chị đã tin tưởng sử dụng dịch vụ ạ.
    ''',
    '''
    Khách hàng: Tôi muốn hỏi tình trạng đơn hàng.
    Nhân viên: Dạ anh/chị cho em xin mã đơn hàng để em kiểm tra ạ.
    ''',
]

for text in examples:
    result = predict_sentiment(text, trainer.model, tokenizer, id2label)
    print("\n" + "-" * 80)
    print(json.dumps(result, ensure_ascii=False, indent=2))


--------------------------------------------------------------------------------
{
  "input": "\n    Khách hàng: Tôi đặt hàng 5 ngày rồi vẫn chưa nhận được, rất bực mình.\n    Nhân viên: Dạ em xin lỗi anh/chị vì sự chậm trễ này. Em sẽ kiểm tra đơn hàng và phản hồi lại ngay ạ.\n    ",
  "normalized_input": "khách hàng: tôi đặt hàng 5 ngày rồi vẫn chưa nhận được, rất bực mình. nhân viên: dạ em xin lỗi anh/chị vì sự chậm trễ này. em sẽ kiểm tra đơn hàng và phản hồi lại ngay ạ.",
  "label": "negative",
  "label_id": 0,
  "confidence": 0.5375033617019653,
  "probabilities": {
    "negative": 0.5375033617019653,
    "neutral": 0.31840214133262634,
    "positive": 0.14409448206424713
  }
}

--------------------------------------------------------------------------------
{
  "input": "\n    Khách hàng: Cảm ơn shop, nhân viên hỗ trợ rất nhanh và nhiệt tình.\n    Nhân viên: Dạ em cảm ơn anh/chị đã tin tưởng sử dụng dịch vụ ạ.\n    ",
  "normalized_input": "khách hàng: cảm ơn shop, nhân viên hỗ 

## 18. Nhập hội thoại thủ công để dự đoán

Chạy cell này rồi nhập hội thoại của bạn.

In [20]:
while True:
    text = input("\nNhập hội thoại cần đánh giá cảm xúc, hoặc nhập 'exit' để thoát:\n> ")

    if text.strip().lower() == "exit":
        break

    result = predict_sentiment(text, trainer.model, tokenizer, id2label)
    print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "input": "Tại sao tôi đặt hàng 1 tuần rồi vẫn chưa nhận đc hàn? ",
  "normalized_input": "tại sao tôi đặt hàng 1 tuần rồi vẫn chưa nhận đc hàn?",
  "label": "negative",
  "label_id": 0,
  "confidence": 0.4360101521015167,
  "probabilities": {
    "negative": 0.4360101521015167,
    "neutral": 0.3907424509525299,
    "positive": 0.1732473373413086
  }
}
{
  "input": "tôi muốn đổi qua size khác lớn hơn",
  "normalized_input": "tôi muốn đổi qua size khác lớn hơn",
  "label": "positive",
  "label_id": 2,
  "confidence": 0.36921757459640503,
  "probabilities": {
    "negative": 0.2824731767177582,
    "neutral": 0.3483092188835144,
    "positive": 0.36921757459640503
  }
}
{
  "input": "sản phẩm này ko vừa với tôi",
  "normalized_input": "sản phẩm này ko vừa với tôi",
  "label": "negative",
  "label_id": 0,
  "confidence": 0.9137967824935913,
  "probabilities": {
    "negative": 0.9137967824935913,
    "neutral": 0.0758429616689682,
    "positive": 0.010360334068536758
  }
}
{
  "input"

## 19. Load lại model đã train để dùng sau này

Sau khi train xong, bạn có thể chạy cell này ở một notebook khác để load model mà không cần train lại.

In [21]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = Path("models/sentiment_phobert/final_model")

loaded_tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), use_fast=False)
loaded_model = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_DIR),
    use_safetensors=(MODEL_DIR / "model.safetensors").exists(),
)

with open(MODEL_DIR / "label_mapping.json", "r", encoding="utf-8") as f:
    mapping = json.load(f)

loaded_id2label = {int(k): v for k, v in mapping["id2label"].items()}

device = "cuda" if torch.cuda.is_available() else "cpu"
loaded_model.to(device)
loaded_model.eval()

sample_text = '''
Khách hàng: Tôi rất thất vọng vì đơn hàng giao sai sản phẩm.
Nhân viên: Dạ em xin lỗi anh/chị, em sẽ kiểm tra và hỗ trợ đổi hàng ngay ạ.
'''

result = predict_sentiment(sample_text, loaded_model, loaded_tokenizer, loaded_id2label)
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "input": "\nKhách hàng: Tôi rất thất vọng vì đơn hàng giao sai sản phẩm.\nNhân viên: Dạ em xin lỗi anh/chị, em sẽ kiểm tra và hỗ trợ đổi hàng ngay ạ.\n",
  "normalized_input": "khách hàng: tôi rất thất vọng vì đơn hàng giao sai sản phẩm. nhân viên: dạ em xin lỗi anh/chị, em sẽ kiểm tra và hỗ trợ đổi hàng ngay ạ.",
  "label": "negative",
  "label_id": 0,
  "confidence": 0.9062438607215881,
  "probabilities": {
    "negative": 0.9062438607215881,
    "neutral": 0.06842619180679321,
    "positive": 0.025329960510134697
  }
}


## 20. Ghi chú quan trọng

### Vì sao dùng PhoBERT?

PhoBERT là model BERT được huấn luyện cho tiếng Việt, phù hợp với các bài toán phân loại văn bản tiếng Việt.

### Vì sao chưa dùng BiLSTM-CRF?

Bài toán sentiment là **text classification**:

```text
Một câu/hội thoại → một nhãn cảm xúc
```

CRF thường dùng cho bài toán gán nhãn từng token như NER:

```text
Một câu → nhãn cho từng từ/token
```

Vì vậy, nếu muốn làm baseline RNN, nên dùng:

```text
BiLSTM + Dense/Softmax
```

hoặc:

```text
BiLSTM + Attention + Dense
```

thay vì BiLSTM-CRF.